# 01 - Historical NFL Data Collection

## Goal

This notebook collects every dataset required to build the NFL Season Projection Model.

Rather than downloading spreadsheets manually, every dataset is collected programmatically so the project can be reproduced and updated in future seasons.

## Data Sources

- nflverse
- NFL schedules
- Team rosters
- Coaching information
- Play-by-play data
- Team statistics
- Advanced metrics

## Output

Processed datasets are saved into the `/data` directory for use throughout the project.

In [58]:
import pandas as pd
import numpy as np
import polars as pl

from pathlib import Path

## Project Directory

In [59]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

ROSTER_DIR = DATA_DIR / "roster"

COACHING_DIR = DATA_DIR / "coaching"

SCHEDULE_DIR = DATA_DIR / "schedules"

print(PROJECT_ROOT)

c:\Users\efriedman\Desktop\NFL-Season-Projections


## Historical Seasons

The first version of the model uses NFL data from 2015 through 2025.

This provides enough recent history to identify patterns while keeping the data relevant to the modern NFL. Older seasons may be added later for specific coaching, roster, and rule change analysis.

In [60]:
SEASONS = list(range(2015, 2026))

print(SEASONS)
print(f"Number of seasons: {len(SEASONS)}")

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of seasons: 11


In [61]:
import nflreadpy as nfl

print("nflreadpy loaded")

nflreadpy loaded


## Download Historical Play-by-Play Data

The full historical dataset is collected for the 2015–2025 seasons.

In [66]:
for season in SEASONS:
    output_path = RAW_DIR / f"play_by_play_{season}.parquet"

    if output_path.exists():
        print(f"{season}: file already exists — skipping")
        continue

    print(f"{season}: downloading...")

    season_pbp = nfl.load_pbp(seasons=[season])
    season_pbp.write_parquet(output_path)

    print(
        f"{season}: saved {season_pbp.height:,} rows "
        f"and {season_pbp.width} columns"
    )

2015: file already exists — skipping
2016: file already exists — skipping
2017: file already exists — skipping
2018: file already exists — skipping
2019: file already exists — skipping
2020: file already exists — skipping
2021: file already exists — skipping
2022: file already exists — skipping
2023: file already exists — skipping
2024: file already exists — skipping
2025: file already exists — skipping


# Schedule and Game Results Data

Schedule data provides the game-level results and context required to calculate team records, point differential, home and road performance, one score results, rest, and future schedule difficulty.

The 2015–2025 schedules are collected separately from play-by-play data because each row represents one NFL game rather than one play.

In [67]:
schedules = nfl.load_schedules(seasons=SEASONS)

print(type(schedules))
print(schedules.shape)
print(schedules.columns[:25])

<class 'polars.dataframe.frame.DataFrame'>
(3028, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline']


In [68]:
schedules.select([
    "game_id",
    "season",
    "game_type",
    "week",
    "gameday",
    "away_team",
    "away_score",
    "home_team",
    "home_score",
    "location",
    "overtime",
    "away_rest",
    "home_rest"
]).head(10)

game_id,season,game_type,week,gameday,away_team,away_score,home_team,home_score,location,overtime,away_rest,home_rest
str,i32,str,i32,str,str,i32,str,i32,str,i32,i32,i32
"""2015_01_PIT_NE""",2015,"""REG""",1,"""2015-09-10""","""PIT""",21,"""NE""",28,"""Home""",0,7,7
"""2015_01_IND_BUF""",2015,"""REG""",1,"""2015-09-13""","""IND""",14,"""BUF""",27,"""Home""",0,7,7
"""2015_01_GB_CHI""",2015,"""REG""",1,"""2015-09-13""","""GB""",31,"""CHI""",23,"""Home""",0,7,7
"""2015_01_KC_HOU""",2015,"""REG""",1,"""2015-09-13""","""KC""",27,"""HOU""",20,"""Home""",0,7,7
"""2015_01_CAR_JAX""",2015,"""REG""",1,"""2015-09-13""","""CAR""",20,"""JAX""",9,"""Home""",0,7,7
"""2015_01_CLE_NYJ""",2015,"""REG""",1,"""2015-09-13""","""CLE""",10,"""NYJ""",31,"""Home""",0,7,7
"""2015_01_SEA_STL""",2015,"""REG""",1,"""2015-09-13""","""SEA""",31,"""STL""",34,"""Home""",1,7,7
"""2015_01_MIA_WAS""",2015,"""REG""",1,"""2015-09-13""","""MIA""",17,"""WAS""",10,"""Home""",0,7,7
"""2015_01_NO_ARI""",2015,"""REG""",1,"""2015-09-13""","""NO""",19,"""ARI""",31,"""Home""",0,7,7


In [69]:
schedule_output_path = SCHEDULE_DIR / "nfl_schedules_2015_2025.parquet"

schedules.write_parquet(schedule_output_path)

print(f"Saved {schedules.height:,} games to:")
print(schedule_output_path)

Saved 3,028 games to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\schedules\nfl_schedules_2015_2025.parquet


# Historical Roster Data

Roster data will be used to measure team continuity, returning starters, age, experience, position group composition, offseason additions and departures, and projected player availability.

Historical rosters are collected separately from play-by-play data because they describe who was on each team rather than what happened on each play.

In [70]:
rosters = nfl.load_rosters(seasons=SEASONS)

print(type(rosters))
print(rosters.shape)
print(rosters.columns[:30])

<class 'polars.dataframe.frame.DataFrame'>
(33195, 36)
['season', 'team', 'position', 'depth_chart_position', 'jersey_number', 'status', 'full_name', 'first_name', 'last_name', 'birth_date', 'height', 'weight', 'college', 'gsis_id', 'espn_id', 'sportradar_id', 'yahoo_id', 'rotowire_id', 'pff_id', 'pfr_id', 'fantasy_data_id', 'sleeper_id', 'years_exp', 'headshot_url', 'ngs_position', 'week', 'game_type', 'status_description_abbr', 'football_name', 'esb_id']


In [71]:
rosters.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB")
).select([
    "season",
    "week",
    "team",
    "full_name",
    "position",
    "depth_chart_position",
    "status",
    "years_exp"
]).head(20)

season,week,team,full_name,position,depth_chart_position,status,years_exp
i32,i32,str,str,str,str,str,i32
2025,19,"""GB""","""Dante Barnett""","""DL""","""DT""","""DEV""",0
2025,19,"""GB""","""Brandon McManus""","""K""","""K""","""ACT""",12
2025,19,"""GB""","""Rashan Gary""","""DL""","""DE""","""ACT""",6
2025,19,"""GB""","""Matthew Orzech""","""LS""","""LS""","""ACT""",6
2025,19,"""GB""","""Keisean Nixon""","""DB""","""CB""","""ACT""",6
…,…,…,…,…,…,…,…
2025,19,"""GB""","""Zayne Anderson""","""DB""","""FS""","""RES""",4
2025,19,"""GB""","""Nate Hobbs""","""DB""","""CB""","""RES""",4
2025,19,"""GB""","""Micah Parsons""","""LB""","""OLB""","""RES""",4


In [72]:
roster_output_path = ROSTER_DIR / "nfl_rosters_2015_2025.parquet"

rosters.write_parquet(roster_output_path)

print(f"Saved {rosters.height:,} roster records to:")
print(roster_output_path)

Saved 33,195 roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\roster\nfl_rosters_2015_2025.parquet


# Historical Player Statistics

Player statistics provide the individual production data needed for quarterback projections, skill position evaluation, defensive player analysis, and position group strength.

Regular season summaries are collected so each player has one season level statistical profile per year.

In [73]:
player_stats = nfl.load_player_stats(
    seasons=SEASONS,
    summary_level="reg"
)

print(type(player_stats))
print(player_stats.shape)
print(player_stats.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(21377, 148)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost']


In [74]:
player_stats.filter(
    (pl.col("season") == 2025) &
    (pl.col("position") == "QB")
).select([
    "player_display_name",
    "recent_team",
    "games",
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "sacks_suffered",
    "passing_epa",
    "passing_cpoe",
    "carries",
    "rushing_yards",
    "rushing_tds"
]).sort("passing_epa", descending=True).head(15)

player_display_name,recent_team,games,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,passing_epa,passing_cpoe,carries,rushing_yards,rushing_tds
str,str,i32,i32,i32,i32,i32,i32,f64,f64,i32,i32,i32
"""Jimmy Garoppolo""","""LA""",3,0,0,0,0,0,null,null,9,-10,0
"""Jarrett Stidham""","""DEN""",1,0,0,0,0,0,null,null,1,-1,0
"""Adrian Martinez""","""SF""",1,0,0,0,0,0,null,null,1,-1,0
"""Jalen Milroe""","""SEA""",3,0,0,0,0,0,null,null,3,4,0
"""Drake Maye""","""NE""",17,492,4394,31,8,47,165.161542,10.781275,103,450,4
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Josh Allen""","""BUF""",16,460,3668,25,10,40,71.326336,3.514859,112,579,14
"""Patrick Mahomes""","""KC""",14,502,3587,22,11,34,68.246941,0.336415,64,422,5
"""Daniel Jones""","""IND""",13,384,3101,19,8,22,65.255515,2.297142,45,164,5


In [75]:
player_stats_output_path = RAW_DIR / "player_stats_2015_2025.parquet"

player_stats.write_parquet(player_stats_output_path)

print(f"Saved {player_stats.height:,} player-season records to:")
print(player_stats_output_path)

Saved 21,377 player-season records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\player_stats_2015_2025.parquet


# Historical Player Participation Data

Participation data helps distinguish roster membership from actual on-field involvement.

This information will later support measures of returning production, starter continuity, position group stability, roster turnover, and player availability. It is particularly important for evaluating offensive line and defensive front continuity, where traditional box score statistics do not fully represent a player's contribution.

In [76]:
PARTICIPATION_SEASONS = list(range(2016, 2026))

participation = nfl.load_participation(
    seasons=PARTICIPATION_SEASONS
)

print(type(participation))
print(participation.shape)
print(participation.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(478989, 26)
['nflverse_game_id', 'old_game_id', 'play_id', 'possession_team', 'offense_formation', 'offense_personnel', 'defenders_in_box', 'defense_personnel', 'number_of_pass_rushers', 'players_on_play', 'offense_players', 'defense_players', 'n_offense', 'n_defense', 'ngs_air_yards', 'time_to_throw', 'was_pressure', 'route', 'defense_man_zone_type', 'defense_coverage_type', 'offense_names', 'defense_names', 'offense_positions', 'defense_positions', 'offense_numbers', 'defense_numbers']


In [77]:
participation.select([
    "nflverse_game_id",
    "play_id",
    "possession_team",
    "offense_formation",
    "offense_personnel",
    "defenders_in_box",
    "defense_personnel",
    "number_of_pass_rushers",
    "time_to_throw",
    "was_pressure",
    "offense_positions",
    "defense_positions"
]).head(10)

nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,time_to_throw,was_pressure,offense_positions,defense_positions
str,f64,str,str,str,i32,str,i32,f64,bool,str,str
"""2016_01_CAR_DEN""",1.0,"""""",null,null,null,null,null,null,null,null,null
"""2016_01_CAR_DEN""",36.0,"""CAR""",null,null,null,null,null,null,null,null,null
"""2016_01_CAR_DEN""",51.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",4,2.323,false,null,null
"""2016_01_CAR_DEN""",75.0,"""DEN""","""I_FORM""","""6 OL, 2 RB, 0 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",6,2.893,true,null,null
"""2016_01_CAR_DEN""",97.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",7,"""4 DL, 2 LB, 5 DB""",3,2.556,false,null,null
"""2016_01_CAR_DEN""",119.0,"""DEN""","""SHOTGUN""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",4,4.59,false,null,null
"""2016_01_CAR_DEN""",143.0,"""DEN""","""SINGLEBACK""","""1 RB, 0 TE, 4 WR""",6,"""4 DL, 2 LB, 5 DB""",5,1.502,false,null,null
"""2016_01_CAR_DEN""",167.0,"""DEN""","""I_FORM""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",null,null,null,null,null
"""2016_01_CAR_DEN""",188.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",null,null,null,null,null


In [78]:
participation_output_path = RAW_DIR / "participation_2016_2025.parquet"

participation.write_parquet(participation_output_path)

print(f"Saved {participation.height:,} participation records to:")
print(participation_output_path)

Saved 478,989 participation records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\participation_2016_2025.parquet


# Historical Snap Count Data

Snap counts measure how frequently individual players were actually on the field.

This information will later help quantify returning production, starter continuity, position group stability, roster turnover, and the importance of player additions and departures.

Snap counts are particularly valuable for offensive line and defensive evaluation because many important contributors are not adequately represented by traditional box score statistics.

In [79]:
snap_counts = nfl.load_snap_counts(seasons=SEASONS)

print(type(snap_counts))
print(snap_counts.shape)
print(snap_counts.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(276948, 16)
['game_id', 'pfr_game_id', 'season', 'game_type', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


In [80]:
snap_counts.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB") &
    (pl.col("position").is_in(["LT", "LG", "C", "RG", "RT", "OL"]))
).select([
    "week",
    "player",
    "position",
    "team",
    "opponent",
    "offense_snaps",
    "offense_pct"
]).sort([
    "week",
    "offense_snaps"
], descending=[False, True]).head(25)

week,player,position,team,opponent,offense_snaps,offense_pct
i32,str,str,str,str,f64,f64
1,"""Elgton Jenkins""","""C""","""GB""","""DET""",48.0,1.0
1,"""Zach Tom""","""OL""","""GB""","""DET""",30.0,0.62
2,"""Elgton Jenkins""","""C""","""GB""","""WAS""",68.0,1.0
3,"""Elgton Jenkins""","""C""","""GB""","""CLE""",65.0,1.0
3,"""Zach Tom""","""OL""","""GB""","""CLE""",1.0,0.02
…,…,…,…,…,…,…
13,"""Jacob Monk""","""C""","""GB""","""DET""",0.0,0.0
14,"""Zach Tom""","""OL""","""GB""","""CHI""",53.0,1.0
14,"""Jacob Monk""","""C""","""GB""","""CHI""",0.0,0.0


In [81]:
snap_counts_output_path = RAW_DIR / "snap_counts_2015_2025.parquet"

snap_counts.write_parquet(snap_counts_output_path)

print(f"Saved {snap_counts.height:,} player-game snap records to:")
print(snap_counts_output_path)

Saved 276,948 player-game snap records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\snap_counts_2015_2025.parquet


# Historical Depth Chart Data

Depth chart data provides information about player roles and position group hierarchy within each team.

This dataset will later help identify starters, backups, roster competitions, positional depth, and changes in expected playing roles from one season to the next.

In [82]:
depth_charts = nfl.load_depth_charts(seasons=SEASONS)

print(type(depth_charts))
print(depth_charts.shape)
print(depth_charts.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(923447, 26)
['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name', 'dt', 'team', 'player_name', 'espn_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']


In [83]:
depth_charts.filter(
    pl.col("season").is_null()
).select([
    "dt",
    "team",
    "club_code",
    "week",
    "game_type",
    "player_name",
    "position",
    "depth_position",
    "pos_grp",
    "pos_name",
    "pos_rank"
]).head(20)

dt,team,club_code,week,game_type,player_name,position,depth_position,pos_grp,pos_name,pos_rank
str,str,str,i32,str,str,str,str,str,str,i32
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Josh Sweat""",null,null,"""Base 4-3 D""","""Left Defensive End""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Roy Lopez""",null,null,"""Base 4-3 D""","""Left Defensive Tackle""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Walter Nolen III""",null,null,"""Base 4-3 D""","""Right Defensive Tackle""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Darius Robinson""",null,null,"""Base 4-3 D""","""Right Defensive End""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Cody Simon""",null,null,"""Base 4-3 D""","""Weakside Linebacker""",1
…,…,…,…,…,…,…,…,…,…,…
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Dante Stills""",null,null,"""Base 4-3 D""","""Right Defensive End""",2
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Owen Pappoe""",null,null,"""Base 4-3 D""","""Weakside Linebacker""",2
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Austin Keys""",null,null,"""Base 4-3 D""","""Middle Linebacker""",2


In [84]:
depth_chart_output_path = RAW_DIR / "depth_charts_raw.parquet"

depth_charts.write_parquet(depth_chart_output_path)

print(f"Saved {depth_charts.height:,} depth chart records to:")
print(depth_chart_output_path)

Saved 923,447 depth chart records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\depth_charts_raw.parquet


# Historical Draft Pick Data

Draft data provides information about incoming rookies and how teams invested draft capital across positions.

This dataset will later support rookie impact estimates, roster turnover analysis, position group investment, and team building evaluation.

In [85]:
draft_picks = nfl.load_draft_picks(seasons=SEASONS)

print(type(draft_picks))
print(draft_picks.shape)
print(draft_picks.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(2821, 36)
['season', 'round', 'pick', 'team', 'gsis_id', 'pfr_player_id', 'cfb_player_id', 'pfr_player_name', 'hof', 'position', 'category', 'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'pass_completions', 'pass_attempts', 'pass_yards', 'pass_tds', 'pass_ints', 'rush_atts', 'rush_yards', 'rush_tds', 'receptions', 'rec_yards', 'rec_tds', 'def_solo_tackles', 'def_ints']


In [86]:
draft_picks.filter(
    pl.col("season") == 2025
).select([
    "round",
    "pick",
    "team",
    "pfr_player_name",
    "position",
    "category",
    "side",
    "college",
    "age"
]).sort("pick").head(20)

round,pick,team,pfr_player_name,position,category,side,college,age
i32,i32,str,str,str,str,str,str,i32
1,1,"""TEN""","""Cam Ward""","""QB""","""QB""","""O""","""Miami (FL)""",23
1,2,"""JAX""","""Travis Hunter""","""WR""","""WR""","""O""","""Colorado""",22
1,3,"""NYG""","""Abdul Carter""","""DE""","""DL""","""D""","""Penn St.""",21
1,4,"""NWE""","""Will Campbell""","""OT""","""OL""","""O""","""LSU""",21
1,5,"""CLE""","""Mason Graham""","""DT""","""DL""","""D""","""Michigan""",22
…,…,…,…,…,…,…,…,…
1,16,"""ARI""","""Walter Nolen""","""DT""","""DL""","""D""","""Mississippi""",21
1,17,"""CIN""","""Shemar Stewart""","""DE""","""DL""","""D""","""Texas A&M""",21
1,18,"""SEA""","""Grey Zabel""","""OT""","""OL""","""O""","""North Dakota St.""",23


In [87]:
draft_output_path = RAW_DIR / "draft_picks_2015_2025.parquet"

draft_picks.write_parquet(draft_output_path)

print(f"Saved {draft_picks.height:,} draft picks to:")
print(draft_output_path)

Saved 2,821 draft picks to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\draft_picks_2015_2025.parquet


# Historical Injury Data

Injury data provides context about player availability and missed time across NFL seasons.

This dataset will later support durability measures, returning player adjustments, roster continuity, position group availability, and estimates of how injuries affected team performance.

Injuries will be treated as context rather than assumed to be perfectly predictable.

In [88]:
injuries = nfl.load_injuries(seasons=SEASONS)

print(type(injuries))
print(injuries.shape)
print(injuries.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(60788, 17)
['season', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified', 'season_type']


In [89]:
injuries.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB")
).select([
    "week",
    "full_name",
    "position",
    "report_primary_injury",
    "report_secondary_injury",
    "report_status",
    "practice_primary_injury",
    "practice_secondary_injury",
    "practice_status"
]).sort([
    "week",
    "full_name"
]).head(30)

week,full_name,position,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status
f64,str,str,str,str,str,str,str,str
1.0,"""Barryn Sorrell""","""DE""","""Knee""",null,"""Questionable""","""Knee""",null,"""Limited Participation in Pract…"
1.0,"""Dontayvion Wicks""","""WR""","""Calf""",null,"""Questionable""","""Calf""",null,"""Limited Participation in Pract…"
1.0,"""Elgton Jenkins""","""C""",null,null,null,"""Hip""",null,"""Limited Participation in Pract…"
1.0,"""Jayden Reed""","""WR""","""Foot""",null,"""Questionable""","""Foot""",null,"""Limited Participation in Pract…"
1.0,"""Jordan Love""","""QB""",null,null,null,"""Thumb""",null,"""Full Participation in Practice"""
…,…,…,…,…,…,…,…,…
3.0,"""Jayden Reed""","""WR""","""Foot""","""Shoulder""","""Out""","""Foot""","""Shoulder""","""Did Not Participate In Practic…"
3.0,"""Jordan Love""","""QB""",null,null,null,"""Thumb""",null,"""Full Participation in Practice"""
3.0,"""Josh Jacobs""","""RB""",null,null,null,"""Ankle""",null,"""Limited Participation in Pract…"


In [90]:
injury_output_path = RAW_DIR / "injuries_2015_2025.parquet"

injuries.write_parquet(injury_output_path)

print(f"Saved {injuries.height:,} injury records to:")
print(injury_output_path)

Saved 60,788 injury records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\injuries_2015_2025.parquet


# Historical Contract Data

Contract data provides additional context about roster construction and team investment in individual players and position groups.

This information can later help identify significant offseason additions and departures, positional spending, roster investment, and the relative importance of personnel changes.

Contract value will be treated as contextual information rather than a direct measure of player ability.

In [91]:
contracts = nfl.load_contracts()

print(type(contracts))
print(contracts.shape)
print(contracts.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(51967, 26)
['player', 'position', 'team', 'is_active', 'year_signed', 'years', 'value', 'apy', 'guaranteed', 'apy_cap_pct', 'inflated_value', 'inflated_apy', 'inflated_guaranteed', 'player_page', 'otc_id', 'gsis_id', 'height', 'weight', 'college', 'draft_year', 'draft_round', 'draft_overall', 'draft_team', 'date_of_birth', 'season_history', 'contract_history']


In [92]:
contracts.select([
    "player",
    "position",
    "team",
    "is_active",
    "year_signed",
    "years",
    "value",
    "apy",
    "guaranteed",
    "apy_cap_pct",
    "draft_year",
    "draft_round",
    "draft_overall",
    "season_history"
]).head(20)

player,position,team,is_active,year_signed,years,value,apy,guaranteed,apy_cap_pct,draft_year,draft_round,draft_overall,season_history
str,str,str,bool,i32,i32,f64,f64,f64,f64,i32,i32,i32,list[struct[13]]
"""Joe Burrow""","""QB""","""Bengals""",true,2023,5,275.0,55.0,146.51,0.245,2020,1,1,"[{""2020"",""Bengals"",0.61,5.970025,0.0,0.0,0.61,6.580025,0.032,24.4901,null,null,null}, {""2020"",""Bengals"",0.61,5.970025,0.0,0.0,0.61,6.580025,0.032,24.4901,null,null,null}, … {""Total"",""Total"",312.299,75.965582,90.0,11.515036,78.825018,339.44306,null,340.694136,null,null,null}]"
"""Aaron Rodgers""","""QB""","""NYJ/GB""",false,2022,3,150.815,50.271667,101.415,0.241,2005,1,24,"[{""2005"",""Packers"",0.23,0.3,null,0.62,0.62,1.15,0.013,2.35,0.0,0.0,null}, {""2005"",""Packers"",0.23,0.3,null,0.62,0.62,1.15,0.013,2.35,0.0,0.0,null}, … {""Total"",""Total"",113.015,145.821136,null,60.78,95.51,328.516136,null,418.312794,6.6,2.3,null}]"
"""Josh Allen""","""QB""","""Bills""",false,2021,6,258.0,43.0,100.0,0.236,2018,1,7,"[{""2018"",""Bills"",0.48,3.371461,0.0,0.0,0.48,3.851461,0.02,13.965844,0.0,null,null}, {""2018"",""Bills"",0.48,3.371461,0.0,0.0,0.48,3.851461,0.02,13.965844,0.0,null,null}, … {""Total"",""Total"",343.08,183.650255,136.94,70.23719,69.22719,482.904439,null,479.404439,4.0,null,null}]"
"""Russell Wilson""","""QB""","""Broncos""",false,2022,5,245.0,49.0,124.0,0.235,2012,3,75,"[{""2012"",""Seahawks"",0.39,0.154868,0.0,0.0,0.0,0.544868,0.005,1.009472,0.0,null,0.0}, {""2012"",""Seahawks"",0.39,0.154868,0.0,0.0,0.0,0.544868,0.005,1.009472,0.0,null,0.0}, … {""Total"",""Total"",86.105651,104.879766,4.0,5.0,84.492,206.579535,null,278.050123,0.5,null,0.529412}]"
"""Dak Prescott""","""QB""","""Cowboys""",true,2024,4,240.0,60.0,129.0,0.235,2016,4,135,"[{""2016"",""Cowboys"",0.45,0.095848,null,null,0.0,0.545848,0.003,0.833392,null,null,null}, {""2016"",""Cowboys"",0.45,0.095848,null,null,0.0,0.545848,0.003,0.833392,null,null,null}, … {""Total"",""Total"",467.589,276.628781,null,null,89.050667,425.759448,null,435.437392,null,null,null}]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Tua Tagovailoa""","""QB""","""Dolphins""",false,2024,4,212.4,53.1,93.171,0.208,2020,1,5,"[{""2020"",""Dolphins"",0.61,4.894625,0.0,0.0,0.61,5.504625,0.025,20.1885,0.0,0.0,null}, {""2020"",""Dolphins"",0.61,4.894625,0.0,0.0,0.61,5.504625,0.025,20.1885,0.0,0.0,null}, … {""Total"",""Total"",30.681,36.3785,5.0,7.401936,36.867936,80.329083,null,125.529083,0.25,0.75,null}]"
"""Jared Goff""","""QB""","""Lions""",true,2024,4,212.0,53.0,113.611832,0.208,2016,1,1,"[{""2016"",""Rams"",0.45,4.629577,null,0.0,0.45,5.079577,0.033,18.968308,0.0,null,null}, {""2016"",""Rams"",0.45,4.629577,null,0.0,0.45,5.079577,0.033,18.968308,0.0,null,null}, … {""Total"",""Total"",162.004196,123.318308,null,53.5,122.314196,376.907504,null,399.107504,1.0,null,null}]"
"""Josh Allen""","""QB""","""Bills""",true,2025,6,330.0,55.0,147.0,0.197,2018,1,7,"[{""2018"",""Bills"",0.48,3.371461,0.0,0.0,0.48,3.851461,0.02,13.965844,0.0,null,null}, {""2018"",""Bills"",0.48,3.371461,0.0,0.0,0.48,3.851461,0.02,13.965844,0.0,null,null}, … {""Total"",""Total"",343.08,183.650255,136.94,70.23719,69.22719,482.904439,null,479.404439,4.0,null,null}]"


In [93]:
contracts_output_path = RAW_DIR / "contracts_raw.parquet"

contracts.write_parquet(contracts_output_path)

print(f"Saved {contracts.height:,} contract records to:")
print(contracts_output_path)

Saved 51,967 contract records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\contracts_raw.parquet


# Historical Trade Data

Trade data provides direct information about significant player and draft pick movement between NFL teams.

This dataset will later help identify major offseason personnel changes and quantify how talent and draft capital moved between organizations. Trade information will be combined with player performance, snap counts, position, and roster data so that the impact of each move can be evaluated in context.

In [95]:
trades = nfl.load_trades()

print(type(trades))
print(trades.shape)
print(trades.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(4975, 11)
['trade_id', 'season', 'trade_date', 'gave', 'received', 'pick_season', 'pick_round', 'pick_number', 'conditional', 'pfr_id', 'pfr_name']


In [100]:
trades.filter(
    pl.col("season") >= 2025
).select([
    "trade_id",
    "season",
    "trade_date",
    "gave",
    "received",
    "pfr_name",
    "pick_season",
    "pick_round",
    "pick_number",
    "conditional"
]).sort(
    ["season", "trade_date"],
    descending=[True, True]
).head(30)

trade_id,season,trade_date,gave,received,pfr_name,pick_season,pick_round,pick_number,conditional
i32,i32,date,str,str,str,i32,i32,i32,i32
2062,2026,2026-04-17,"""ATL""","""JAX""","""Ruke Orhorhoro""",null,null,null,null
2062,2026,2026-04-17,"""JAX""","""ATL""","""Maason Smith""",null,null,null,null
2061,2026,2026-04-10,"""GB""","""PHI""","""Dontayvion Wicks""",null,null,null,null
2061,2026,2026-04-10,"""PHI""","""GB""","""""",2026,5,null,0
2061,2026,2026-04-10,"""PHI""","""GB""","""""",2027,6,null,0
…,…,…,…,…,…,…,…,…,…
2055,2026,2026-03-11,"""TEN""","""DAL""","""""",2026,7,null,0
2053,2026,2026-03-11,"""DAL""","""SF""","""Osa Odighizuwa""",null,null,null,null
2052,2026,2026-03-10,"""NYJ""","""LV""","""""",2026,6,null,0


In [102]:
trades_output_path = RAW_DIR / "trades_raw.parquet"

trades.write_parquet(trades_output_path)

print(f"Saved {trades.height:,} trade records to:")
print(trades_output_path)

Saved 4,975 trade records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\trades_raw.parquet


# Historical Weekly Roster Data

Weekly roster data provides a more detailed view of how team personnel changes throughout a season.

This dataset will later help identify player movement, roster additions and departures, status changes, practice squad activity, and changes in team depth over time.

Weekly rosters will be compared across seasons to help measure roster continuity and offseason turnover.

In [103]:
weekly_rosters = nfl.load_rosters_weekly(seasons=SEASONS)

print(type(weekly_rosters))
print(weekly_rosters.shape)
print(weekly_rosters.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(498381, 36)
['season', 'team', 'position', 'depth_chart_position', 'jersey_number', 'status', 'full_name', 'first_name', 'last_name', 'birth_date', 'height', 'weight', 'college', 'gsis_id', 'espn_id', 'sportradar_id', 'yahoo_id', 'rotowire_id', 'pff_id', 'pfr_id', 'fantasy_data_id', 'sleeper_id', 'years_exp', 'headshot_url', 'ngs_position', 'week', 'game_type', 'status_description_abbr', 'football_name', 'esb_id', 'gsis_it_id', 'smart_id', 'entry_year', 'rookie_year', 'draft_club']


In [104]:
weekly_rosters.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB")
).select([
    "week",
    "full_name",
    "position",
    "depth_chart_position",
    "status",
    "status_description_abbr",
    "years_exp",
    "gsis_id"
]).sort([
    "week",
    "full_name"
]).head(30)

week,full_name,position,depth_chart_position,status,status_description_abbr,years_exp,gsis_id
i32,str,str,str,str,str,i32,str
1,"""Aaron Banks""","""OL""","""G""","""ACT""","""A01""",4,"""00-0036551"""
1,"""Anthony Belton""","""OL""","""T""","""ACT""","""A01""",0,"""00-0040726"""
1,"""Arron Mosby""","""DL""","""DE""","""DEV""","""P06""",3,"""00-0037364"""
1,"""Barryn Sorrell""","""DL""","""DE""","""INA""","""A01""",0,"""00-0040010"""
1,"""Ben Sims""","""TE""","""TE""","""INA""","""A01""",2,"""00-0038809"""
…,…,…,…,…,…,…,…
1,"""Elgton Jenkins""","""OL""","""G""","""ACT""","""A01""",6,"""00-0035526"""
1,"""Emanuel Wilson""","""RB""","""RB""","""ACT""","""A01""",2,"""00-0038797"""
1,"""Evan Williams""","""DB""","""FS""","""ACT""","""A01""",1,"""00-0039813"""


In [105]:
weekly_rosters_output_path = RAW_DIR / "weekly_rosters_2015_2025.parquet"

weekly_rosters.write_parquet(weekly_rosters_output_path)

print(f"Saved {weekly_rosters.height:,} weekly roster records to:")
print(weekly_rosters_output_path)

Saved 498,381 weekly roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\weekly_rosters_2015_2025.parquet


In [106]:
data_inventory = pl.DataFrame({
    "dataset": [
        "Play-by-Play",
        "Schedules",
        "Season Rosters",
        "Player Statistics",
        "Participation",
        "Snap Counts",
        "Depth Charts",
        "Draft Picks",
        "Injuries",
        "Contracts",
        "Trades",
        "Weekly Rosters",
    ],
    "rows": [
        sum(
            pl.read_parquet(RAW_DIR / f"play_by_play_{season}.parquet").height
            for season in SEASONS
        ),
        schedules.height,
        rosters.height,
        player_stats.height,
        participation.height,
        snap_counts.height,
        depth_charts.height,
        draft_picks.height,
        injuries.height,
        contracts.height,
        trades.height,
        weekly_rosters.height,
    ],
    "primary_purpose": [
        "Team efficiency and play-level performance",
        "Game results, rest, opponents, and schedule context",
        "Season roster composition and player attributes",
        "Individual player production and efficiency",
        "Personnel, formations, pressure, and on-field participation",
        "Playing time and returning production",
        "Expected roles, starters, and position-group depth",
        "Rookie additions and draft capital",
        "Player availability and injury context",
        "Roster investment and contract context",
        "Player and draft-pick movement between teams",
        "Weekly roster status and continuity",
    ]
})

data_inventory

dataset,rows,primary_purpose
str,i64,str
"""Play-by-Play""",532376,"""Team efficiency and play-level…"
"""Schedules""",3028,"""Game results, rest, opponents,…"
"""Season Rosters""",33195,"""Season roster composition and …"
"""Player Statistics""",21377,"""Individual player production a…"
"""Participation""",478989,"""Personnel, formations, pressur…"
…,…,…
"""Draft Picks""",2821,"""Rookie additions and draft cap…"
"""Injuries""",60788,"""Player availability and injury…"
"""Contracts""",51967,"""Roster investment and contract…"
